# 03 — New model: ConvNeXt-Base

Default config: `convnext_legacy.yaml`, the recipe that actually produced the 92.77% run (gradual unfreezing, local normalization, RandAugment + Cutout, legacy dropout), re-measured over seeds. New recipes are new YAML files: change `CONFIG_PATH`.

Protocol (Phase 0): seeded runs, split check + fingerprint, checkpoint selected on validation, test reported with the val-selected weights and with the last epoch, binary metrics as in the paper. All hyperparameters live in the YAML config.

**Kaggle, once per notebook:** Settings → Accelerator *GPU T4 x2*, Internet *On*; *Add Input* → the dataset (already split into train/val/test); `GITHUB_TOKEN` in Add-ons → Secrets only if the repo is private. For multi-seed runs use *Save Version → Save & Run All*.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO, BRANCH = "caMatt99/Cloud_detection_project", "main"
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/input").is_dir()


def sh(cmd, secret=None):
    """Runs a shell command, raises on failure; `secret` is masked in any message."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    mask = (lambda s: s.replace(secret, "***")) if secret else (lambda s: s)
    if r.returncode:
        raise RuntimeError(f"{mask(cmd)}\n{mask(r.stdout + r.stderr)}")
    return r.stdout.strip()


if ON_KAGGLE:
    # Code in /tmp so it stays out of the notebook output; results/checkpoints go to /kaggle/working.
    PROJECT_ROOT = Path("/tmp/Cloud_detection_project")
    if PROJECT_ROOT.exists():
        sh(f"git -C {PROJECT_ROOT} pull -q origin {BRANCH}")
    else:
        try:  # GITHUB_TOKEN is only needed if the repo is private
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            token = None
        auth = f"{token}@" if token else ""
        sh(f"git clone -q -b {BRANCH} https://{auth}github.com/{REPO}.git {PROJECT_ROOT}", secret=token)
    try:
        import timm   # noqa: F401
    except ImportError:
        sh(f"{sys.executable} -m pip install -q timm")
else:
    # Local: no automatic pull, so uncommitted edits are never touched.
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(sh("git log -1 --format='commit %h  %s'"))   # the commit this run uses

from utils import print_env
print_env()
# NB: after a git pull that changes src/, restart the kernel — modules already imported are not reloaded.

In [ ]:
from utils import load_config
from dataset import check_split

CONFIG_PATH = "configs/convnext_legacy.yaml"
SEEDS = None          # None -> experiment.seeds from the config; e.g. [0] for a quick check

cfg = load_config(CONFIG_PATH)
print(f"experiment : {cfg['experiment']['name']}  seeds={SEEDS or cfg['experiment']['seeds']}")
print(f"dataset    : {cfg['dataset']['path']}")
print(f"split      : fingerprint {check_split(cfg['dataset']['path'])}")

## Train over seeds

One W&B run per seed, grouped by `experiment.name`. With two GPUs `run_seeds_parallel` starts one worker per GPU (seeds round-robin, logs in `<results>/<experiment>/logs/`); with one GPU it runs the seeds sequentially.

In [ ]:
from experiment import run_seeds_parallel

table = run_seeds_parallel(CONFIG_PATH, SEEDS)

## Comparison with the paper

`accuracy` in %, the rest binary (positive = `true`) in [0, 1]. 1 test image = 0.21 pp.

In [ ]:
import pandas as pd
from evaluate import PAPER_RESNET50

mean, std = table.loc["mean"], table.loc["std"]
fmt = lambda tag, k: f"{mean[f'test_{tag}_{k}']:.4f} ± {std[f'test_{tag}_{k}']:.4f}"
pd.DataFrame({
    "paper ResNet50 (single run)": PAPER_RESNET50,
    "val-selected (mean ± std)":   {k: fmt("best", k) for k in PAPER_RESNET50},
    "last epoch (mean ± std)":     {k: fmt("last", k) for k in PAPER_RESNET50},
})

## Download results

Zip of `<results>/<experiment>/`: `summary.csv`, one JSON per seed (metrics + full history), `predictions_seed<k>.csv` (per-image P(true), val + test), worker logs. Checkpoints are not included.

In [ ]:
from experiment import download_link

download_link(cfg)